In [0]:
from pyspark.sql.functions import *

# Load Silver sources
order_items = spark.table("ecommerce_dev.silver.order_items")
orders = spark.table("ecommerce_dev.silver.orders")

order_items.printSchema()
order_items.show(5)
print("order_items rows:", order_items.count())

root
 |-- order_id: string (nullable = false)
 |-- order_item_id: integer (nullable = false)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+--------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|  bronze_ingested_at|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+--------------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|2026-08-04 00:55:...|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61

In [0]:
dim_customer = spark.table("ecommerce_dev.gold.dim_customer")
dim_seller   = spark.table("ecommerce_dev.gold.dim_seller").filter(col("is_current") == True)
dim_product  = spark.table("ecommerce_dev.gold.dim_product").filter(col("is_current") == True)
dim_date     = spark.table("ecommerce_dev.gold.dim_date")

fact_order_items = (
    order_items
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp",
                         "order_status"), "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_seller.select("seller_key", "seller_id"), "seller_id", "left")
    .join(dim_product.select("product_key", "product_id"), "product_id", "left")
    .join(dim_date.select(col("date_key").alias("order_date_key"),
                           col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .select(
        "order_id",
        "order_item_id",
        "customer_key",
        "seller_key",
        "product_key",
        "order_date_key",
        "order_status",
        "price",
        "freight_value"
    )
)

print("Fact rows:", fact_order_items.count())
print("Null customer_key:", fact_order_items.filter(col("customer_key").isNull()).count())
print("Null seller_key:", fact_order_items.filter(col("seller_key").isNull()).count())
print("Null product_key:", fact_order_items.filter(col("product_key").isNull()).count())
print("Null order_date_key:", fact_order_items.filter(col("order_date_key").isNull()).count())

Fact rows: 112650
Null customer_key: 0
Null seller_key: 0
Null product_key: 0
Null order_date_key: 0


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.fact_order_items (
    fact_order_item_key BIGINT GENERATED ALWAYS AS IDENTITY,
    order_id STRING,
    order_item_id INT,
    customer_key BIGINT,
    seller_key BIGINT,
    product_key BIGINT,
    order_date_key INT,
    order_status STRING,
    price DECIMAL(10,2),
    freight_value DECIMAL(10,2)
) USING DELTA
""")

fact_order_items.write.format("delta").mode("append").saveAsTable("ecommerce_dev.gold.fact_order_items")

spark.sql("ALTER TABLE ecommerce_dev.gold.fact_order_items ALTER COLUMN fact_order_item_key SET NOT NULL")
spark.sql("ALTER TABLE ecommerce_dev.gold.fact_order_items ADD CONSTRAINT pk_fact_order_items PRIMARY KEY (fact_order_item_key)")

spark.sql("""
COMMENT ON TABLE ecommerce_dev.gold.fact_order_items IS
'Primary fact table at order_item grain. FKs to Dim_Customer, Dim_Seller (current SCD2), Dim_Product (current SCD2), Dim_Date.'
""")

DataFrame[]